# Building the 2022–2024 Extension: Features from Scratch

## What this notebook does and why it is the most important one

The Kelly et al. (2020) dataset ends in December 2021. Our backtest runs to November 2024. The three extra years — January 2022 to November 2024 — are the extension period, and they are the most valuable part of our out-of-sample test. These 36 months represent data that did not exist when Kelly published, and that no other group can have used to fit their model. They are genuinely new.

To use these 36 months, we need to construct the same 30 signals (7 value + 17 quality + 6 momentum) that the Kelly dataset provided for 1957–2021. We do this from scratch using raw CRSP and Compustat data, applying identical signal definitions. The output is `features_2022_2024.csv` — the extension counterpart to `kelly_final.csv`.

## The two Compustat sources we combine

Several signals require annual accounting data (book equity, total assets, sales, dividends). Others require quarterly data (quarterly ROA, quarterly ROE, accrual volatility over 16 quarters). We pull both:

- **`compustat_annual.csv`** — annual fundamentals pulled in the previous notebook
- **`compustat_quarterly.csv`** — quarterly fundamentals pulled in this notebook from `comp.fundq`

## The 6-month availability lag

For annual Compustat, a firm with December 31 fiscal year-end files its 10-K approximately 60–90 days later. After auditing, printing, and SEC processing, we apply a conservative 6-month lag: `available_date = datadate + 6 months`. Data with `datadate = 2021-12-31` is only used from July 2022 onwards. This is the Fama-French convention and is non-negotiable for preventing look-ahead bias.

For quarterly data, we use the earnings announcement date (`rdq`) rather than a fixed lag. Firms typically announce quarterly earnings 30–45 days after quarter-end, so `rdq` is more precise than a blanket 3-month lag. Where `rdq` is missing, we fall back to `datadate + 3 months`.

## Memory management

The cross-join of CRSP × Compustat, if done naively, would produce billions of rows. We avoid this by:
1. Pre-filtering both sides to only relevant gvkeys and years before merging
2. Processing the annual merge firm-by-firm in a loop
3. Expanding quarterly data to a monthly panel before merging, so the final join is 1-to-1 on `(gvkey, date)`
4. Freeing memory aggressively with `del` and `gc.collect()` after each step

## Step 1 — Verify feature definitions against the Kelly dataset

Before constructing anything, I verify that all 30 signal names we intend to compute actually exist in the Kelly dataset. If any are missing, there is a naming inconsistency that would cause a silent mismatch when the two datasets are concatenated in the backtest pipeline.

I also check coverage rates from the start of `kelly_final.csv` (early rows, 1957) to understand the baseline NaN rates for each signal. This is purely diagnostic — the actual coverage check that matters is the one at the end of this notebook on `features_2022_2024.csv`.

In [ ]:
import pandas as pd

# Load just the first 1000 rows of Kelly to check column names and coverage
# No need to load 3.9M rows just for a schema verification
kelly = pd.read_csv('kelly_final.csv', nrows=1000)
print(kelly.columns.tolist())

# These are the exact signal names as they appear in Kelly
# Our extension features must use identical names for the backtest pipeline to work
VALUE = [
    'bm',       # book to market
    'bm_ia',    # industry-adjusted book to market
    'cfp',      # cashflow to price
    'cfp_ia',   # industry-adjusted cashflow to price
    'ep',       # earnings to price
    'sp',       # sales to price
    'dy',       # dividend yield
]

QUALITY = [
    'roaq',     # quarterly ROA
    'roeq',     # quarterly ROE
    'gma',      # gross profitability (Novy-Marx)
    'operprof', # operating profitability
    'acc',      # accruals — Sloan (1996), lower = more cash-backed earnings
    'absacc',   # absolute accruals
    'pctacc',   # accruals scaled by earnings
    'stdacc',   # standard deviation of quarterly accruals over 16 quarters
    'agr',      # asset growth — overinvestment predictor
    'egr',      # equity growth
    'lgr',      # liability growth
    'lev',      # leverage
    'ms',       # Mohanram S-score (8-signal composite)
    'nincr',    # number of consecutive quarterly earnings increases (max 8)
    'tb',       # tax-to-book — Lev and Nissim, taxable income cannot be managed
    'cashdebt', # cashflow to debt coverage
]

MOMENTUM = [
    'mom1m',    # 1-month reversal
    'mom6m',    # 6-month momentum (skipping month t-1)
    'mom12m',   # 12-month momentum — Jegadeesh-Titman
    'mom36m',   # 36-month long-term reversal — De Bondt-Thaler
    'chmom',    # change in 12-month momentum (acceleration)
    'indmom',   # industry momentum — Moskowitz-Grinblatt
]

# Verify all signal names exist in Kelly — any mismatch here would break the pipeline
all_features = VALUE + QUALITY + MOMENTUM
missing = [f for f in all_features if f not in kelly.columns]
present = [f for f in all_features if f in kelly.columns]

print(f"Features present in Kelly: {len(present)}/{len(all_features)}")
print(f"Missing: {missing}")

# Coverage from the first 1000 rows — early rows will have lower coverage
# due to limited Compustat history in the 1950s-1960s
print("\nValue feature coverage:")
for f in VALUE:
    if f in kelly.columns:
        print(f"  {f}: {kelly[f].notna().mean():.1%}")

print("\nQuality feature coverage:")
for f in QUALITY:
    if f in kelly.columns:
        print(f"  {f}: {kelly[f].notna().mean():.1%}")

print("\nMomentum feature coverage:")
for f in MOMENTUM:
    if f in kelly.columns:
        print(f"  {f}: {kelly[f].notna().mean():.1%}")

## Step 2 — Check coverage from the middle of the Kelly dataset

The first 1000 rows of Kelly cover 1957 — before Compustat had broad coverage. To get a realistic picture of what coverage looks like in the main training period, I sample from around row 1,000,000 which corresponds to the 1990s. This is more representative of what the extension period coverage will look like.

In [ ]:
# Skip the first 1M rows to sample from the middle of the dataset
# This gives a more representative coverage picture than the early 1957 rows
kelly = pd.read_csv('kelly_final.csv', skiprows=range(1, 1000000), nrows=1000)

print("Date range of sample:", kelly['date'].min(), "to", kelly['date'].max())

print("\nValue feature coverage:")
for f in VALUE:
    if f in kelly.columns:
        print(f"  {f}: {kelly[f].notna().mean():.1%}")

print("\nQuality feature coverage:")
for f in QUALITY:
    if f in kelly.columns:
        print(f"  {f}: {kelly[f].notna().mean():.1%}")

print("\nMomentum feature coverage:")
for f in MOMENTUM:
    if f in kelly.columns:
        print(f"  {f}: {kelly[f].notna().mean():.1%}")

## Step 3 — Pull quarterly Compustat

Quarterly fundamentals come from `comp.fundq`. We need quarterly data for several signals:
- `roaq` and `roeq` — quarterly ROA and ROE, computed from quarterly income and balance sheet items
- `stdacc` — standard deviation of quarterly accruals over 16 rolling quarters
- `ms` and `nincr` — the Mohanram S-score and earnings consistency signals that compare sequential quarters

The key field here is `rdq` — the earnings announcement date. This is the date the firm publicly released its quarterly results. We use this as the `available_date` for quarterly data rather than applying a blanket 3-month lag, because it is the actual date the information became public. Where `rdq` is missing we fall back to `datadate + 3 months`.

We pull from 1950 to ensure we have the 16-quarter rolling history needed for `stdacc` fully populated by 2022.

In [ ]:
import wrds
conn = wrds.Connection(wrds_username='muhab')

# Load the link table and standardise gvkey format
# Zero-padding to 6 digits is critical — inconsistency here silently drops firms
link   = pd.read_csv('link_table.csv', parse_dates=['linkdt', 'linkenddt'])
gvkeys = link['gvkey'].astype(str).str.strip().str.zfill(6).unique().tolist()
print(f"Pulling quarterly data for {len(gvkeys)} firms...")

chunk_size = 500
chunks     = [gvkeys[i:i+chunk_size] for i in range(0, len(gvkeys), chunk_size)]

all_chunks = []
for i, chunk in enumerate(chunks):
    print(f"  Chunk {i+1}/{len(chunks)}...")
    gvkey_str = ','.join([f"'{g}'" for g in chunk])

    q = conn.raw_sql(f"""
        SELECT
            gvkey,
            datadate,
            fyearq,
            fqtr,
            rdq,        -- earnings announcement date: when the data became public
            ibq,        -- quarterly income before extraordinary items (for roaq, roeq, nincr)
            saleq,      -- quarterly sales
            atq,        -- quarterly total assets
            ceqq,       -- quarterly common equity
            seqq,       -- quarterly stockholders equity
            niq,        -- quarterly net income
            oiadpq,     -- quarterly operating income after depreciation (for ms signal)
            oibdpq,     -- quarterly operating income before depreciation
            dpq,        -- quarterly depreciation
            actq,       -- quarterly current assets (for accrual calculation)
            lctq,       -- quarterly current liabilities
            cheq,       -- quarterly cash and equivalents
            dlcq,       -- quarterly current portion of long-term debt
            dlttq,      -- quarterly long-term debt
            cshoq,      -- quarterly shares outstanding (for dilution signal)
            prccq,      -- quarterly price per share
            xrdq        -- quarterly R&D expense (for ms signal)
        FROM comp.fundq
        WHERE datadate BETWEEN '1950-01-01' AND '2024-12-31'
            AND gvkey IN ({gvkey_str})
            AND indfmt = 'INDL'
            AND datafmt = 'STD'
            AND popsrc = 'D'
            AND consol = 'C'
    """, date_cols=['datadate', 'rdq'])

    all_chunks.append(q)

compq = pd.concat(all_chunks, ignore_index=True)
compq = compq.sort_values(['gvkey', 'datadate'])
# Keep only the most recent entry for each (gvkey, datadate) pair
# Restated quarterly filings replace prior entries
compq = compq.drop_duplicates(subset=['gvkey', 'datadate'], keep='last')

print(f"\nQuarterly rows: {len(compq):,}")
print(f"Date range: {compq['datadate'].min()} to {compq['datadate'].max()}")

compq.to_csv('compustat_quarterly.csv', index=False)
print("Saved: compustat_quarterly.csv")

conn.close()

## Step 4 — Re-pull annual Compustat with txt included

The initial Compustat annual pull was missing `txt` (income tax expense), which is required for the tax-to-book signal (`tb`). Rather than modifying the previous notebook and re-running everything, I re-pull here and overwrite `compustat_annual.csv`. The logic is otherwise identical to the previous notebook.

In [ ]:
# Re-pull annual Compustat adding txt (income tax expense) which was missing
# txt is required for the tax-to-book signal: tb = (txt - ΔTXP) / |IB|

import wrds
import pandas as pd

conn = wrds.Connection(wrds_username='muhab')

link = pd.read_csv('link_table.csv')
link['gvkey'] = link['gvkey'].astype(str).str.strip().str.zfill(6)
gvkeys = link['gvkey'].unique().tolist()

chunk_size = 500
chunks     = [gvkeys[i:i+chunk_size] for i in range(0, len(gvkeys), chunk_size)]

all_chunks = []
for i, chunk in enumerate(chunks):
    print(f"  Chunk {i+1}/{len(chunks)}...")
    gvkey_str = ','.join([f"'{g}'" for g in chunk])

    q = conn.raw_sql(f"""
        SELECT
            gvkey, datadate, fyear, fyr,
            at, lt, seq, ceq,
            pstk, pstkrv, pstkl,
            txditc, txp, txt,
            act, lct, che, rect, invt,
            dlc, dltt,
            ni, ib, oiadp, oibdp,
            dp, sale, cogs, xsga, xrd,
            capx, dv, csho, prcc_f,
            naicsh, sich
        FROM comp.funda
        WHERE datadate BETWEEN '1950-01-01' AND '2024-12-31'
            AND gvkey IN ({gvkey_str})
            AND indfmt = 'INDL'
            AND datafmt = 'STD'
            AND popsrc = 'D'
            AND consol = 'C'
            AND at > 0
    """, date_cols=['datadate'])

    all_chunks.append(q)

comp = pd.concat(all_chunks, ignore_index=True)
comp = comp.sort_values(['gvkey', 'datadate'])
comp = comp.drop_duplicates(subset=['gvkey', 'datadate'], keep='last')

print(f"Rows: {len(comp):,}")
print(f"txt coverage: {comp['txt'].notna().mean():.1%}")

comp.to_csv('compustat_annual.csv', index=False)
print("Saved: compustat_annual.csv")
conn.close()

## Step 5 — Build the full feature panel for 2022–2024

This is the main computation cell. It runs eight sequential steps:

**Step 5.1 — Prepare CRSP for the extension period**  
Filter CRSP to the extension universe (permnos active from 2022 onwards) and pull data from 2017 onwards to provide momentum signal history (mom36m needs 3 years of prior returns). Join the CCM link table to add gvkeys to each CRSP row, filtering to date-valid links only.

**Step 5.2 — Pre-filter annual Compustat**  
Filter to the relevant gvkeys and years (2014+). Compute the `available_date` (datadate + 6 months) and `next_available` (the next annual filing's available date) for each firm-year. These two dates define the window during which each annual observation is the most recent valid accounting data.

**Step 5.3 — Pre-filter quarterly Compustat**  
Same logic for quarterly data, using `rdq` as the availability date and 2018+ to ensure 16-quarter history for `stdacc`.

**Step 5.4 — Expand quarterly to a monthly panel**  
Rather than doing a range-join merge (which would explode to billions of rows), I first convert the quarterly observations into a monthly panel. For each firm, for each quarterly observation, I identify every calendar month where that observation is the most recent available, and create one row per month. The resulting panel is then joinable 1-to-1 on `(gvkey, date)`.

**Step 5.5 — Merge annual Compustat firm-by-firm**  
Loop through each gvkey, merge the CRSP rows with the annual Compustat rows using the `available_date`/`next_available` window, and concatenate results. This avoids the cross-join explosion.

**Step 5.6 — Merge quarterly panel**  
Simple 1-to-1 join on `(gvkey, date)` since the quarterly panel is already expanded to monthly frequency.

**Step 5.7 — Compute all 30 features**  
The `compute_all_features` function constructs every signal from the merged panel using the same formulas as Kelly et al.

**Step 5.8 — Filter to 2022+ and save**  
The panel was built from 2017 to provide history for lagged computations. The final output contains only 2022–2024 rows.

In [ ]:
import pandas as pd
import numpy as np
import gc

print("Loading files...")
crsp  = pd.read_csv('crsp_returns.csv', parse_dates=['date'])
comp  = pd.read_csv('compustat_annual.csv', parse_dates=['datadate'])
compq = pd.read_csv('compustat_quarterly.csv',
                     parse_dates=['datadate', 'rdq'])
link  = pd.read_csv('link_table.csv',
                     parse_dates=['linkdt', 'linkenddt'])

# Standardise gvkey format across all files — inconsistency here silently drops firms
for df in [comp, compq, link]:
    df['gvkey'] = df['gvkey'].astype(str).str.strip().str.zfill(6)
link['permno'] = link['permno'].astype(int)

# ============================================================
# STEP 5.1: PREPARE CRSP FOR THE EXTENSION PERIOD
# Pull from 2017 to provide 5 years of return history
# needed for mom36m (3 years) and other lagged momentum signals
# ============================================================

ext_permnos = crsp[crsp['date'] >= '2022-01-01']['permno'].unique()
crsp_ext = crsp[
    (crsp['permno'].isin(ext_permnos)) &
    (crsp['date'] >= '2017-01-01')
].copy()

# Add gvkey to CRSP rows via the CCM link table
# Filter to date-valid links only — a firm's gvkey can change over time
crsp_ext = crsp_ext.merge(
    link[['permno', 'gvkey', 'linkdt', 'linkenddt']],
    on='permno', how='left'
)
crsp_ext = crsp_ext[
    (crsp_ext['date'] >= crsp_ext['linkdt']) &
    (crsp_ext['date'] <= crsp_ext['linkenddt'])
].copy()
crsp_ext = crsp_ext.dropna(subset=['gvkey'])
crsp_ext['gvkey'] = crsp_ext['gvkey'].astype(str).str.strip().str.zfill(6)

del link
gc.collect()

print(f"CRSP ext rows: {len(crsp_ext):,}")
ext_gvkeys = set(crsp_ext['gvkey'].unique())

# ============================================================
# STEP 5.2: PRE-FILTER AND PREPARE ANNUAL COMPUSTAT
# Only keep gvkeys we need and years that could be relevant
# For 2022-2024 CRSP data with 6-month lag, we need accounting from 2014+
# ============================================================

comp_clean = comp[
    (comp['gvkey'].isin(ext_gvkeys)) &
    (comp['datadate'] >= '2014-01-01')
].copy()

print(f"Filtered annual comp: {len(comp_clean):,} rows")

# Apply the 6-month availability lag — this is the Fama-French convention
# A firm with December year-end is only available from July of the following year
comp_clean['available_date'] = (
    (comp_clean['datadate'] + pd.DateOffset(months=6))
    .dt.to_period('M').dt.to_timestamp()
)
comp_clean = comp_clean.sort_values(['gvkey', 'available_date'])
# next_available tells us when this observation becomes stale (replaced by the next filing)
comp_clean['next_available'] = (
    comp_clean.groupby('gvkey')['available_date']
    .shift(-1).fillna(pd.Timestamp('2099-12-31'))
)

del comp
gc.collect()

# ============================================================
# STEP 5.3: PRE-FILTER AND PREPARE QUARTERLY COMPUSTAT
# Need data from 2018+ to have 16 quarters of history for stdacc by 2022
# ============================================================

compq_clean = compq[
    (compq['gvkey'].isin(ext_gvkeys)) &
    (compq['datadate'] >= '2018-01-01')
].copy()

print(f"Filtered quarterly comp: {len(compq_clean):,} rows")

# Use rdq (earnings announcement date) as the availability date
# This is the actual date the information became public — more precise than a fixed lag
# Fall back to datadate + 3 months where rdq is missing
compq_clean['available_date'] = (
    compq_clean['rdq']
    .fillna(compq_clean['datadate'] + pd.DateOffset(months=3))
    .dt.to_period('M').dt.to_timestamp()
)
compq_clean = compq_clean.sort_values(['gvkey', 'available_date'])
compq_clean['next_available_q'] = (
    compq_clean.groupby('gvkey')['available_date']
    .shift(-1).fillna(pd.Timestamp('2099-12-31'))
)

del compq
gc.collect()

# ============================================================
# STEP 5.4: EXPAND QUARTERLY TO MONTHLY PANEL
# Convert quarterly snapshots to a monthly panel BEFORE merging
# This turns the range-join problem into a simple 1-to-1 join on (gvkey, date)
# avoiding the cross-join explosion that would otherwise occur
# ============================================================

print("Expanding quarterly to monthly panel...")

date_range = pd.date_range(start='2017-01-01', end='2024-11-01', freq='MS')

q_cols = ['ibq', 'atq', 'ceqq', 'niq', 'oiadpq', 'dpq',
          'actq', 'lctq', 'cheq', 'dlcq', 'dlttq', 'xrdq']

monthly_q_list = []

for gvkey, firm_data in compq_clean.groupby('gvkey'):
    firm_rows = []
    for _, row in firm_data.iterrows():
        # Find all months where this quarterly observation is the most recent available
        valid_months = date_range[
            (date_range >= row['available_date']) &
            (date_range <  row['next_available_q'])
        ]
        if len(valid_months) == 0:
            continue
        for month in valid_months:
            firm_rows.append({
                'gvkey': gvkey,
                'date':  month,
                **{col: row[col] for col in q_cols if col in row.index}
            })
    if firm_rows:
        monthly_q_list.append(pd.DataFrame(firm_rows))

monthly_q = pd.concat(monthly_q_list, ignore_index=True)
# One row per (gvkey, date) — the 1-to-1 join is now safe
monthly_q = monthly_q.drop_duplicates(subset=['gvkey', 'date'], keep='last')

print(f"Monthly quarterly panel: {len(monthly_q):,} rows")

del compq_clean, monthly_q_list
gc.collect()

# ============================================================
# STEP 5.5: MERGE ANNUAL COMPUSTAT — FIRM BY FIRM
# Loop prevents the cross-join explosion
# Each merge is small: one firm's CRSP rows × that firm's annual filings
# ============================================================

print("Merging annual Compustat...")

ann_chunks = []

for gvkey, crsp_firm in crsp_ext.groupby('gvkey'):
    comp_firm = comp_clean[comp_clean['gvkey'] == gvkey]
    if len(comp_firm) == 0:
        # Keep the CRSP rows even without accounting data — they will have NaN features
        ann_chunks.append(crsp_firm)
        continue

    merged_firm = crsp_firm.merge(comp_firm, on='gvkey', how='left')
    # Apply the availability window: only use accounting data that was publicly available
    # at the time of portfolio formation
    merged_firm = merged_firm[
        (merged_firm['date'] >= merged_firm['available_date']) &
        (merged_firm['date'] <  merged_firm['next_available'])
    ]
    # Take the most recent annual observation for any duplicate (permno, date)
    merged_firm = merged_firm.drop_duplicates(subset=['permno', 'date'], keep='last')
    ann_chunks.append(merged_firm)

merged = pd.concat(ann_chunks, ignore_index=True)

del ann_chunks, comp_clean
gc.collect()

print(f"After annual merge: {len(merged):,}")
print(f"Annual match rate: {merged['at'].notna().mean():.2%}")

# ============================================================
# STEP 5.6: MERGE QUARTERLY — simple 1-to-1 join
# Safe because the quarterly panel was pre-expanded to monthly frequency
# ============================================================

print("Merging quarterly...")
merged = merged.merge(monthly_q, on=['gvkey', 'date'], how='left')

del monthly_q
gc.collect()

print(f"After quarterly merge: {len(merged):,}")
print(f"Quarterly match rate: {merged['atq'].notna().mean():.2%}")

# ============================================================
# STEP 5.7: COMPUTE ALL 30 FEATURES
# Identical definitions to Kelly et al.
# safe_div handles zero denominators and NaN propagation cleanly
# ============================================================

def safe_div(a, b):
    """Division that returns NaN rather than inf or error on zero/NaN denominators."""
    a = pd.to_numeric(a, errors='coerce')
    b = pd.to_numeric(b, errors='coerce')
    return np.where((b != 0) & b.notna() & a.notna(), a / b, np.nan)


def compute_all_features(df):
    out = df.copy()
    out = out.sort_values(['permno', 'date']).reset_index(drop=True)

    # Book equity: follows Fama-French definition
    # Start with stockholders equity (seq), fall back to ceq + preferred stock,
    # then to assets minus liabilities. Subtract preferred stock (redemption value
    # preferred over liquidation value over carrying value). Add deferred taxes.
    out['be'] = (
        out['seq']
        .fillna(out['ceq'] + out['pstk'].fillna(0))
        .fillna(out['at'] - out['lt'])
    )
    out['be'] = (out['be']
                 - out['pstkrv']
                   .fillna(out['pstkl'])
                   .fillna(out['pstk'])
                   .fillna(0))
    out['be'] = out['be'] + out['txditc'].fillna(0)
    # Floor at a small positive number to avoid division by zero in bm
    out['be'] = out['be'].clip(lower=0.001)

    # 2-digit SIC code for industry-adjusted signals
    out['sic2'] = out['siccd'].astype(str).str[:2]

    # ── VALUE SIGNALS ────────────────────────────────────────
    out['bm']  = safe_div(out['be'], out['me'])
    out['ep']  = safe_div(out['ib'], out['me'])
    out['cfp'] = safe_div(out['ib'].fillna(0) + out['dp'].fillna(0), out['me'])
    out['sp']  = safe_div(out['sale'], out['me'])
    out['dy']  = safe_div(out['dv'], out['me'])
    # Industry-adjusted: subtract the median within (date, 2-digit SIC) each month
    ind_med_bm  = out.groupby(['date', 'sic2'])['bm'].transform('median')
    ind_med_cfp = out.groupby(['date', 'sic2'])['cfp'].transform('median')
    out['bm_ia']  = out['bm']  - ind_med_bm
    out['cfp_ia'] = out['cfp'] - ind_med_cfp

    # ── QUALITY — PROFITABILITY ───────────────────────────────
    out['roaq']     = safe_div(out['ibq'], out['atq'])
    out['roeq']     = safe_div(out['ibq'], out['ceqq'])
    out['gma']      = safe_div(out['sale'] - out['cogs'].fillna(0), out['at'])
    out['operprof'] = safe_div(out['oiadp'] - out['xsga'].fillna(0), out['be'])

    # ── QUALITY — ACCRUALS ────────────────────────────────────
    # Sloan (1996) accruals: change in working capital minus depreciation, scaled by assets
    # Lower accruals = more cash-backed earnings = higher quality
    out['act_lag']  = out.groupby('permno')['act'].shift(1)
    out['lct_lag']  = out.groupby('permno')['lct'].shift(1)
    out['at_lag1']  = out.groupby('permno')['at'].shift(1)
    out['avg_at']   = (out['at'] + out['at_lag1']) / 2
    out['delta_wc'] = (
        (out['act'] - out['che'].fillna(0)) -
        (out['lct'] - out['dlc'].fillna(0))
    ) - (
        (out['act_lag'] - out['che'].fillna(0)) -
        (out['lct_lag'] - out['dlc'].fillna(0))
    )
    out['acc']    = safe_div(out['delta_wc'] - out['dp'].fillna(0), out['avg_at'])
    out['absacc'] = out['acc'].abs()
    out['pctacc'] = safe_div(out['acc'], out['ni'].abs())
    # Tax-to-book: taxable income (proxied by tax expense) cannot be managed
    # High tb = earnings are backed by real taxable income
    out['txp_lag'] = out.groupby('permno')['txp'].shift(1)
    out['tb']      = safe_div(
        out['txt'].fillna(0) - (out['txp'].fillna(0) - out['txp_lag'].fillna(0)),
        out['ib'].abs()
    )
    out['cashdebt'] = safe_div(
        out['ib'].fillna(0) + out['dp'].fillna(0),
        out['dltt'].fillna(0) + out['dlc'].fillna(0)
    )
    # Quarterly accrual volatility: needs 16 quarterly observations (4 years)
    # This is why we pull quarterly data back to 2018 for the 2022 extension
    out['acc_q']  = safe_div(out['ibq'].fillna(0) - out['oiadpq'].fillna(0), out['atq'])
    out['stdacc'] = out.groupby('permno')['acc_q'].transform(
        lambda x: x.rolling(16, min_periods=8).std()
    )

    # ── QUALITY — GROWTH / INVESTMENT ────────────────────────
    # Cooper et al. (2008): firms that over-invest in assets subsequently underperform
    out['agr']     = safe_div(out['at'] - out['at_lag1'], out['at_lag1'])
    out['ceq_lag'] = out.groupby('permno')['ceq'].shift(1)
    out['egr']     = safe_div(out['ceq'] - out['ceq_lag'], out['ceq_lag'])
    out['lt_lag']  = out.groupby('permno')['lt'].shift(1)
    out['lgr']     = safe_div(out['lt'] - out['lt_lag'], out['lt_lag'])
    out['ppegt']     = out['at'] - out['act'].fillna(0) - out['che'].fillna(0)
    out['ppegt_lag'] = out.groupby('permno')['ppegt'].shift(1)
    out['invt_lag']  = out.groupby('permno')['invt'].shift(1)
    out['invest']    = safe_div(
        (out['ppegt'] - out['ppegt_lag'].fillna(0)) +
        (out['invt'].fillna(0) - out['invt_lag'].fillna(0)),
        out['at_lag1']
    )

    # ── QUALITY — FINANCIAL HEALTH ────────────────────────────
    out['debt'] = out['dltt'].fillna(0) + out['dlc'].fillna(0)
    out['lev']  = safe_div(out['debt'], out['at'])

    # Mohanram S-score: 8 binary signals, scored 0 or 1 each
    # Compares the firm to the industry median on each dimension
    out['cfo_at']     = safe_div(out['oiadpq'], out['atq'])
    ind_med_roa       = out.groupby(['date', 'sic2'])['roaq'].transform('median')
    ind_med_cfo       = out.groupby(['date', 'sic2'])['cfo_at'].transform('median')
    out['xrd_lag']    = out.groupby('permno')['xrd'].shift(1)
    out['capx_lag']   = out.groupby('permno')['capx'].shift(1)
    out['ato']        = safe_div(out['sale'], out['at'])
    out['ato_lag']    = out.groupby('permno')['ato'].shift(1)
    out['margin']     = safe_div(out['ib'], out['sale'])
    out['margin_lag'] = out.groupby('permno')['margin'].shift(1)
    out['csho_lag']   = out.groupby('permno')['csho'].shift(1)

    s1 = (out['roaq']   > ind_med_roa).astype(float)
    s2 = (out['cfo_at'] > ind_med_cfo).astype(float)
    s3 = (out['oiadpq'].fillna(0) > out['ibq'].fillna(0)).astype(float)
    s4 = (out['xrd'].fillna(0)    > out['xrd_lag'].fillna(0)).astype(float)
    s5 = (out['capx'].fillna(0)   > out['capx_lag'].fillna(0)).astype(float)
    s6 = (out['ato']    > out['ato_lag']).astype(float)
    s7 = (out['margin'] > out['margin_lag']).astype(float)
    s8 = (out['csho']   <= out['csho_lag']).astype(float)  # no dilution = positive
    out['ms'] = s1 + s2 + s3 + s4 + s5 + s6 + s7 + s8

    # Earnings consistency: how many of the last 8 quarters showed YoY improvement
    out['ib_q_lag'] = out.groupby('permno')['ibq'].shift(1)
    out['earn_up']  = (out['ibq'] > out['ib_q_lag']).astype(float)
    out['nincr']    = out.groupby('permno')['earn_up'].transform(
        lambda x: x.rolling(8, min_periods=1).sum()
    )

    # ── MOMENTUM SIGNALS ─────────────────────────────────────
    # mom1m: last month's return — known to reverse, included as a control
    out['mom1m']  = out.groupby('permno')['ret'].shift(1)
    # mom6m: 6-month return, skipping month t-1 to avoid 1-month reversal contamination
    out['mom6m']  = out.groupby('permno')['ret'].transform(
        lambda x: x.shift(2).rolling(5)
        .apply(lambda r: (1 + r).prod() - 1, raw=True)
    )
    # mom12m: 12-month Jegadeesh-Titman momentum, skipping month t-1
    out['mom12m'] = out.groupby('permno')['ret'].transform(
        lambda x: x.shift(2).rolling(11)
        .apply(lambda r: (1 + r).prod() - 1, raw=True)
    )
    # mom36m: 36-month long-term reversal — De Bondt and Thaler (1985)
    out['mom36m'] = out.groupby('permno')['ret'].transform(
        lambda x: x.shift(13).rolling(24)
        .apply(lambda r: (1 + r).prod() - 1, raw=True)
    )
    # chmom: change in 12-month momentum — captures acceleration or deceleration
    out['mom12m_lag6'] = out.groupby('permno')['mom12m'].shift(6)
    out['chmom']  = out['mom12m'] - out['mom12m_lag6']
    # indmom: average 12-month momentum within the 2-digit SIC industry
    out['indmom'] = out.groupby(['date', 'sic2'])['mom12m'].transform('mean')

    return out


print("\nComputing features...")
panel = compute_all_features(merged)

del merged
gc.collect()

# ============================================================
# STEP 5.8: FILTER TO 2022+ AND SAVE
# The full panel starts from 2017 to build momentum and lag history
# Only the 2022-2024 rows are the actual extension output
# ============================================================

VALUE    = ['bm', 'bm_ia', 'cfp', 'cfp_ia', 'ep', 'sp', 'dy']
QUALITY  = ['roaq', 'roeq', 'gma', 'operprof', 'acc', 'absacc',
            'pctacc', 'tb', 'cashdebt', 'stdacc', 'agr', 'egr',
            'lgr', 'invest', 'lev', 'ms', 'nincr']
MOMENTUM = ['mom1m', 'mom6m', 'mom12m', 'mom36m', 'chmom', 'indmom']
ALL_FEATURES = VALUE + QUALITY + MOMENTUM

panel_ext = panel[panel['date'] >= '2022-01-01'].copy()

output_cols = ['permno', 'date', 'ret', 'me', 'sic2'] + ALL_FEATURES
output_cols = [c for c in output_cols if c in panel_ext.columns]
panel_ext   = panel_ext[output_cols]

print(f"\nShape: {panel_ext.shape}")
print(f"Date range: {panel_ext['date'].min()} to {panel_ext['date'].max()}")
print(f"Unique permnos: {panel_ext['permno'].nunique():,}")

# Coverage check — flag signals below 60% coverage for review
# Quality signals will naturally have lower coverage than value/momentum
print("\nFeature coverage:")
for f in ALL_FEATURES:
    if f in panel_ext.columns:
        cov  = panel_ext[f].notna().mean()
        flag = " <-- LOW" if cov < 0.6 else ""
        print(f"  {f}: {cov:.1%}{flag}")

panel_ext.to_csv('features_2022_2024.csv', index=False)
print("\nSaved: features_2022_2024.csv")

## Step 6 — Final verification: confirm feature sets match across both datasets

The backtest pipeline concatenates `kelly_final.csv` and `features_2022_2024.csv` and runs on the combined dataset. For this to work correctly, both files must have identical column names for all 30 signals. This cell confirms there are no naming mismatches before the pipeline runs.

It also prints the exact feature counts per model, which is what appears on our methodology slide:
- Model A — Value only: 7 features
- Model B — Value + Quality: 24 features  
- Model C — Value + Quality + Momentum: 30 features
- Model D — Value + Momentum: 13 features

In [ ]:
VALUE    = ['bm', 'bm_ia', 'cfp', 'cfp_ia', 'ep', 'sp', 'dy']

QUALITY  = ['roaq', 'roeq', 'gma', 'operprof',
            'acc', 'absacc', 'pctacc',
            'tb', 'cashdebt', 'stdacc',
            'agr', 'egr', 'lgr', 'invest',
            'lev', 'ms', 'nincr']

MOMENTUM = ['mom1m', 'mom6m', 'mom12m', 'mom36m', 'chmom', 'indmom']

# The four model feature sets — these map directly to our presentation slides
VALUE_ONLY        = VALUE                         # Model A: 7 features
VALUE_QUALITY     = VALUE + QUALITY               # Model B: 24 features
VALUE_QUALITY_MOM = VALUE + QUALITY + MOMENTUM    # Model C: 30 features
VALUE_MOM         = VALUE + MOMENTUM              # Model D: 13 features

print(f"Model A — Value only:              {len(VALUE_ONLY)} features")
print(f"Model B — Value + Quality:         {len(VALUE_QUALITY)} features")
print(f"Model C — Value + Quality + Mom:   {len(VALUE_QUALITY_MOM)} features")
print(f"Model D — Value + Momentum:        {len(VALUE_MOM)} features")

# Final cross-check: both datasets must have all required features with identical names
# Any mismatch here would cause silent NaN columns in the backtest
import pandas as pd

kelly = pd.read_csv('kelly_final.csv', nrows=5)
ext   = pd.read_csv('features_2022_2024.csv', nrows=5)

for name, cols in [('VALUE', VALUE),
                    ('QUALITY', QUALITY),
                    ('MOMENTUM', MOMENTUM)]:
    missing_k = [f for f in cols if f not in kelly.columns]
    missing_e = [f for f in cols if f not in ext.columns]
    print(f"\n{name}:")
    print(f"  Missing from Kelly:     {missing_k if missing_k else 'none'}")
    print(f"  Missing from extension: {missing_e if missing_e else 'none'}")